<a href="https://colab.research.google.com/github/AndrewEisele/Accessibility_App/blob/main/accessibility_app.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Accessibility App
My app helps users decide which game in each genre is most accessible by sorting available Steam games by price, platform, required age, genre, and language while creating a cumulative score out of 10.

In [12]:
from google.colab import drive
drive.mount("/content/drive", force_remount=True)

import glob
import os
import time
import pandas as pd

# Look for steam_games.csv in My Drive and in folders up to 3 levels down
found = []
for pattern in ["", "*/", "*/*/", "*/*/*/"]:
    found = found + glob.glob("/content/drive/MyDrive/" + pattern + "steam_games.csv")

if len(found) == 0:
    print("❌ Can't find steam_games.csv in your Drive. Upload it next to this notebook and run this cell again.")
else:
    FOLDER = os.path.dirname(found[0]) + "/"
    games = pd.read_csv(FOLDER + "steam_games.csv")
    print(f"✅ Loaded {len(games)} games from:", FOLDER)

Mounted at /content/drive
✅ Loaded 1000 games from: /content/drive/MyDrive/


In [13]:
def most_languages(games, n=10):
  df = games.copy()
  top = df.sort_values("n_languages", ascending=False).head(n)
  return top[["name", "n_languages"]]

most_languages(games)

,name,n_languages
412,Combat Master: Season 3,35
500,DDNet,35
931,Mr.Mine,31
116,American Truck Simulator,30
144,Black Mesa,30
98,Factorio,30
17,Euro Truck Simulator 2,30
0,Counter-Strike 2,29
216,Half-Life: Alyx,28
193,Dota Underlords,28


In [14]:
import sys
import time

import pandas as pd
import requests

csv_file = "steam_games.csv"
top_n = 10
max_lookups = 30

In [21]:
# Gamescorer creates the scoring system for the app

class Gamescorer:
  def __init__(self):
    self.price_cap = 60
    self.age_cap = 18
    self.language_cap = 20

  def price_score(self, price):
    score = 10 * (1 - price / self.price_cap)
    return max(score,0)

  def platform_score(self, windows, mac, linux):
    count = 0
    if windows:
      count = count +1
    if mac:
      count = count +1
    if linux:
      count = count +1
    final_count = 10 * count /3
    return final_count

  def age_score(self, age):
    score = 10 * (1 - age / self.age_cap)
    return max(score, 0)

  def language_score(self, n_languages):
    score = 10 * n_languages / self.language_cap
    return min(score, 10)

  def base_score(self, game):
    total = (
        self.price_score(game["price"])
        + self.platform_score(game["windows"], game["mac"], game["linux"])
        + self.age_score(game["required_age"])
        + self.language_score(game["n_languages"])
    )
    return total / 4

  def bonus(self, game, lowest_price):
    if game["price"] == 0 or lowest_price is None:
      return 0
    if game ["price"] <= lowest_price + 0.01:
      if game["discount"] > 0:
        return 2
      return 1
    return 0


In [22]:
# load_games makes each row of data into a dictionary which is easily called

def load_games(filename):
  table = pd.read_csv(filename)
  games = []
  for i, row in table.iterrows():
    game = {
        "appid": int(row["appid"]),
        "name": str(row["name"]),
        "price": float(row["price"]),
        "discount": float(row["discount"]),
        "required_age": float(row["required_age"]),
        "windows": row["windows"] is True or str(row["windows"]) == "True",
        "mac": row["mac"] is True or str(row["mac"]) == "True",
        "linux": row["linux"] is True or str(row["linux"]) == "True",
        "n_languages": float(row["n_languages"]),
        "genres": [],
    }
    if isinstance(row["genres"], str):
      for genre in row["genres"].split(","):
        game["genres"].append(genre.strip())
    games.append(game)
  return games




In [39]:
# Using Gamescorer

scorer = Gamescorer()
game = games[0]

print("Game:", game["name"])
print("Price score:   ", scorer.price_score(game["price"]))
print("Platform score:", scorer.platform_score(game["windows"], game["mac"], game["linux"]))
print("Age score:     ", scorer.age_score(game["required_age"]))
print("Language score:", scorer.language_score(game["n_languages"]))
print("Base score:    ", round(scorer.base_score(game), 2))

Game: Counter-Strike 2
Price score:    10.0
Platform score: 6.666666666666667
Age score:      10.0
Language score: 10
Base score:     9.17


In [45]:
# Using load_games

games = load_games("steam_games.csv")
print("Games loaded:", len(games))
print("Type of games:", type(games))

print(games[0])
print(games[999])

game_1 = games[0]
print("Name: ", game_1["name"])


Games loaded: 1000
Type of games: <class 'list'>
{'appid': 730, 'name': 'Counter-Strike 2', 'price': 0.0, 'discount': 0.0, 'required_age': 0.0, 'windows': True, 'mac': False, 'linux': True, 'n_languages': 29.0, 'genres': ['Action', 'Free To Play']}
{'appid': 374280, 'name': 'Hired Ops', 'price': 0.0, 'discount': 0.0, 'required_age': 0.0, 'windows': True, 'mac': False, 'linux': False, 'n_languages': 3.0, 'genres': ['Action', 'Free To Play', 'Massively Multiplayer']}
Name:  Counter-Strike 2
